
# 03 - NLP Features & Risk Phrase Analysis

## Goal

Explore the text-feature and risk-phrase layer produced by the Enron surveillance pipeline.

This notebook focuses on:

- Email text feature distributions
- Risk phrase category coverage
- Risk band distribution
- Top risky emails
- Sender-level risk concentration
- Simple surveillance-oriented NLP insights

This is an exploratory notebook. It does **not** modify pipeline outputs.


## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

DATA_PATH = Path("../data/processed/email_risk_scores.parquet")

df = pd.read_parquet(DATA_PATH)

print("Shape:", df.shape)
df.head()


## 2. Column Overview

The risk-score dataset combines parsed email fields, text features, risk phrase flags, network-derived sender features, and final risk scores.


In [ ]:
columns_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str),
    "null_count": df.isna().sum().values,
    "null_pct": (df.isna().mean() * 100).round(2).values
})

columns_df.sort_values("null_pct", ascending=False).head(30)

## 3. Text Feature Summary

In [ ]:
text_feature_cols = [
    "subject_length",
    "body_length",
    "word_count",
    "sentence_count",
    "exclamation_count",
    "question_count",
    "uppercase_ratio",
]

available_text_cols = [col for col in text_feature_cols if col in df.columns]

df[available_text_cols].describe().T

In [ ]:
if "body_length" in df.columns:
    plt.figure(figsize=(10, 4))
    df["body_length"].fillna(0).clip(upper=10000).hist(bins=60)
    plt.title("Email Body Length Distribution, Capped at 10,000 Characters")
    plt.xlabel("Body length")
    plt.ylabel("Email count")
    plt.tight_layout()
    plt.show()

In [ ]:
if "word_count" in df.columns:
    plt.figure(figsize=(10, 4))
    df["word_count"].fillna(0).clip(upper=2000).hist(bins=60)
    plt.title("Email Word Count Distribution, Capped at 2,000 Words")
    plt.xlabel("Word count")
    plt.ylabel("Email count")
    plt.tight_layout()
    plt.show()

## 4. Reply, Forward and Attachment Indicators

In [ ]:
indicator_cols = [
    "has_subject",
    "is_reply",
    "is_forward",
    "has_attachment_reference",
]

available_indicators = [col for col in indicator_cols if col in df.columns]

indicator_summary = []

for col in available_indicators:
    count = int(df[col].fillna(0).sum())
    pct = round(count / len(df) * 100, 2)
    indicator_summary.append({"indicator": col, "email_count": count, "email_pct": pct})

pd.DataFrame(indicator_summary)


## 5. Risk Phrase Category Distribution

The rule-based phrase layer flags emails containing compliance-style risk signals.


In [ ]:
risk_category_cols = [
    "risk_confidentiality",
    "risk_concealment",
    "risk_deletion",
    "risk_urgency_pressure",
    "risk_legal_regulatory",
    "risk_financial_risk",
    "risk_offline_meeting",
]

available_risk_cols = [col for col in risk_category_cols if col in df.columns]

risk_summary = []

for col in available_risk_cols:
    count = int(df[col].fillna(0).sum())
    pct = round(count / len(df) * 100, 2)
    risk_summary.append({
        "risk_category": col.replace("risk_", ""),
        "email_count": count,
        "email_pct": pct
    })

risk_summary_df = (
    pd.DataFrame(risk_summary)
    .sort_values("email_count", ascending=False)
)

risk_summary_df

In [ ]:
if not risk_summary_df.empty:
    plot_df = risk_summary_df.sort_values("email_count")

    plt.figure(figsize=(9, 4))
    plt.barh(plot_df["risk_category"], plot_df["email_count"])
    plt.title("Risk Phrase Category Counts")
    plt.xlabel("Email count")
    plt.ylabel("Risk category")
    plt.tight_layout()
    plt.show()

## 6. Risk Phrase Score Distribution

In [ ]:
if "risk_phrase_score" in df.columns:
    df["risk_phrase_score"].describe()

In [ ]:
if "risk_phrase_score" in df.columns:
    plt.figure(figsize=(9, 4))
    df["risk_phrase_score"].fillna(0).hist(bins=30)
    plt.title("Risk Phrase Score Distribution")
    plt.xlabel("Risk phrase score")
    plt.ylabel("Email count")
    plt.tight_layout()
    plt.show()

## 7. Final Risk Score and Bands

In [ ]:
risk_band_summary = (
    df["risk_band"]
    .value_counts()
    .rename_axis("risk_band")
    .reset_index(name="email_count")
)

risk_band_summary["email_pct"] = (
    risk_band_summary["email_count"] / len(df) * 100
).round(2)

risk_band_summary

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(risk_band_summary["risk_band"], risk_band_summary["email_count"])
plt.title("Risk Band Distribution")
plt.xlabel("Risk band")
plt.ylabel("Email count")
plt.tight_layout()
plt.show()

In [ ]:
if "final_risk_score" in df.columns:
    plt.figure(figsize=(9, 4))
    df["final_risk_score"].hist(bins=50)
    plt.title("Final Risk Score Distribution")
    plt.xlabel("Final risk score")
    plt.ylabel("Email count")
    plt.tight_layout()
    plt.show()

    df["final_risk_score"].describe()


## 8. Top Risky Emails

These are candidate investigation records. A high score should be interpreted as a review-prioritization signal, not proof of misconduct.


In [ ]:
display_cols = [
    "date",
    "from_email",
    "to_email",
    "subject",
    "final_risk_score",
    "risk_band",
    "risk_phrase_score",
    "risk_phrase_category_count",
    "sender_network_volume",
    "sender_betweenness",
]

available_display_cols = [col for col in display_cols if col in df.columns]

top_risky = (
    df.sort_values("final_risk_score", ascending=False)
    [available_display_cols]
    .head(20)
)

top_risky


## 9. Sender-Level Risk Concentration

Aggregate risk at the sender level to identify employees or accounts that may deserve closer review.


In [ ]:
sender_risk = (
    df.groupby("from_email", dropna=True)
    .agg(
        email_count=("message_id", "count"),
        avg_risk_score=("final_risk_score", "mean"),
        max_risk_score=("final_risk_score", "max"),
        medium_high_count=("risk_band", lambda x: x.isin(["Medium", "High"]).sum()),
        high_count=("risk_band", lambda x: (x == "High").sum()),
        risk_phrase_hits=("has_any_risk_phrase", "sum"),
    )
    .reset_index()
)

sender_risk["medium_high_pct"] = (
    sender_risk["medium_high_count"] / sender_risk["email_count"] * 100
).round(2)

sender_risk = sender_risk.sort_values(
    ["high_count", "medium_high_count", "avg_risk_score", "email_count"],
    ascending=False
)

sender_risk.head(20)

In [ ]:
top_sender_plot = (
    sender_risk[sender_risk["email_count"] >= 50]
    .sort_values("medium_high_count", ascending=False)
    .head(15)
    .sort_values("medium_high_count")
)

plt.figure(figsize=(10, 5))
plt.barh(top_sender_plot["from_email"], top_sender_plot["medium_high_count"])
plt.title("Top Senders by Medium/High Risk Email Count")
plt.xlabel("Medium/High risk emails")
plt.ylabel("Sender")
plt.tight_layout()
plt.show()


## 10. Simple Text Examples by Risk Category

Inspect a few subjects associated with each risk category.


In [ ]:
examples = {}

for col in available_risk_cols:
    sample = (
        df[df[col] == 1]
        [["from_email", "to_email", "subject", "risk_phrase_score", "final_risk_score"]]
        .dropna(subset=["subject"])
        .head(5)
    )

    examples[col.replace("risk_", "")] = sample

list(examples.keys())

In [ ]:
# Change category name to inspect examples.
category = "legal_regulatory"

examples.get(category)


## 11. Key Takeaways

This notebook validates that:

- Text features are available for all parsed emails.
- Risk phrase categories generate meaningful coverage.
- Final risk scores are strongly imbalanced, which is expected for surveillance.
- Sender-level aggregation reveals candidates for investigation workflow design.
- The next step is network-focused analysis and dashboard presentation.
